# Relatório de Auditoria e Qualidade dos Dados

**Projeto:** Projeto ANEEL - Energia em Risco: Análise de Dados de Continuidade Elétrica e Previsão de Risco Regulatório (ANEEL)  

**Metodologia:** CRISP-DM

**Período de Análise:** 2021 – 2025

---

## Visão Geral

Este notebook faz parte da **Fase 2 do CRISP-DM - Compreensão dos Dados (Data Understanding)** e valida a conformidade dos dados da camada **interim (`data/interim/`)** com o contrato de dados do projeto ANEEL (2021–2025). O objetivo é verificar schema, tipos, filtros de escopo, nulos, conversões, granularidade, duplicidades e relacionamentos.

---

## Sumário

1. Configurações do ambiente  
2. Arquivos e camadas avaliadas  
3. Regras derivadas do contrato de dados  
4. Validação de schema e tipos  
5. Validação de filtros de escopo  
6. Validação de nulos e conversões  
7. Validação de granularidade e duplicidades  
8. Validação de chaves e relacionamentos  
9. Validações específicas por dataset  
10. Problemas encontrados e ações recomendadas  

---

## 1. Configurações do ambiente

In [ ]:
# Importações
import sys
from pathlib import Path

import duckdb
import pandas as pd

# Configurações de diretórios
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CONT_PATH, INT_PATH, LIM_PATH, ATR_PATH, REG_PATH
from src.data.constants import ANO_INICIO, ANO_FIM

# Configurações de visualização
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda value: f"{value:.2f}")

# Conexão DuckDB
con = duckdb.connect()



## 2. Verificação de arquivos

In [118]:
# Inventário dos arquivos da camada interim
DATASETS = {
    "continuidade": Path(CONT_PATH),
    "interrupcoes": Path(INT_PATH),
    "limites": Path(LIM_PATH),
    "atributos": Path(ATR_PATH),
    "regiao": Path(REG_PATH),
}

inventario = []
for nome, caminho in DATASETS.items():
    existe = caminho.is_file()
    qtde_linhas = int(con.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(caminho)]).fetchone()[0]) if existe else 0
    descricao = {
            row[0]: row[1]
            for row in con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [str(caminho)]).fetchall()
        }
    inventario.append({
        "dataset": nome,
        "camada": "interim",
        "arquivo": caminho.name,
        "existe": existe,
        "tamanho_mb": caminho.stat().st_size / (1024 * 1024) if existe else 0,
        "linhas": qtde_linhas,
        "colunas": len(descricao),
    })

df_inventario = pd.DataFrame(inventario)
display(df_inventario)

,dataset,camada,arquivo,existe,tamanho_mb,linhas,colunas
0,continuidade,interim,continuidade_2021_2025.parquet,True,2.48,375082,8
1,interrupcoes,interim,interrupcoes_2021_2025.parquet,True,694.11,44410820,15
2,limites,interim,limites_2021_2025.parquet,True,0.10,31844,6
3,atributos,interim,atributos.parquet,True,0.10,7569,9
4,regiao,interim,regiao.parquet,True,0.15,15162,5


## 3. Regras derivadas do contrato de dados

As validações reproduzem as regras do quality gate da camada `interim`: existência dos arquivos, verificação de dados nulos, schema mínimo, intervalo de anos, nulos críticos, valores inválidos e duplicidades de continuidade.

O relatório também inclui validações complementares, como conferência de tipos, duplicidades em outras tabelas e consistência dos relacionamentos entre datasets.

In [110]:
# Regras centrais e granularidades esperadas
regras_contrato = pd.DataFrame([
    ["continuidade", "schema/domínio", "DEC/FEC, ano no período, mês 1-12, valor >= 0", "IdeConjunto + AnoIndice + NumPeriodoIndice + SigIndicador"],
    ["interrupcoes", "schema/domínio", "datas válidas, duração >= 0, consumidores >= 0", "uma linha por evento"],
    ["limites", "schema/domínio", "DEC/FEC, ano no período, limite numérico", "NumCNPJ + IdeConjunto + SigIndicador + AnoIndice"],
    ["atributos", "schema/domínio", "IdeConjunto não nulo", "NumCNPJ + IdeConjunto"],
    ["regiao", "schema/domínio", "IdeConjunto não nulo e região compatível com UF", "uma linha por IdeConjunto"],
], columns=["dataset", "categoria", "regra", "granularidade"])

display(
    regras_contrato.style.set_properties(
        subset=["granularidade"],
        **{"white-space": "normal", "min-width": "320px"},
    )
)

,dataset,categoria,regra,granularidade
0,continuidade,schema/domínio,"DEC/FEC, ano no período, mês 1-12, valor >= 0",IdeConjunto + AnoIndice + NumPeriodoIndice + SigIndicador
1,interrupcoes,schema/domínio,"datas válidas, duração >= 0, consumidores >= 0",uma linha por evento
2,limites,schema/domínio,"DEC/FEC, ano no período, limite numérico",NumCNPJ + IdeConjunto + SigIndicador + AnoIndice
3,atributos,schema/domínio,IdeConjunto não nulo,NumCNPJ + IdeConjunto
4,regiao,schema/domínio,IdeConjunto não nulo e região compatível com UF,uma linha por IdeConjunto


## 4. Validação de schema e tipos

In [111]:
# Colunas e tipos mínimos previstos no contrato para cada arquivo interim
SCHEMA_ESPERADO = {
    "continuidade": {
        "AnoIndice": {"INTEGER", "BIGINT"}, "NumPeriodoIndice": {"INTEGER", "BIGINT"},
        "SigAgente": {"VARCHAR"}, "NumCNPJ": {"VARCHAR"}, "IdeConjunto": {"BIGINT", "INTEGER"},
        "DscConjunto": {"VARCHAR"}, "SigIndicador": {"VARCHAR"}, "VlrIndiceEnviado": {"DOUBLE", "FLOAT"},
    },
    "interrupcoes": {
        "AnoIndice": {"INTEGER", "BIGINT"}, "NumPeriodoIndice": {"INTEGER", "BIGINT"},
        "DatInicioInterrupcao": {"TIMESTAMP"}, "DatFimInterrupcao": {"TIMESTAMP"},
        "DuracaoHoras": {"DOUBLE", "FLOAT"}, "SigAgente": {"VARCHAR"}, "NumCNPJ": {"VARCHAR"},
        "IdeConjunto": {"BIGINT", "INTEGER"}, "NumOrdemInterrupcao": {"VARCHAR"},
        "NumConsumidorConjunto": {"BIGINT", "INTEGER"},
    },
    "limites": {
        "SigAgente": {"VARCHAR"}, "NumCNPJ": {"VARCHAR"}, "IdeConjunto": {"BIGINT", "INTEGER"},
        "SigIndicador": {"VARCHAR"}, "AnoIndice": {"INTEGER", "BIGINT"}, "VlrLimite": {"DOUBLE", "FLOAT"},
    },
    "atributos": {
        "DatGeracaoConjuntoDados": {"DATE"}, "SigAgente": {"VARCHAR"}, "NumCNPJ": {"VARCHAR"},
        "IdeConjunto": {"BIGINT", "INTEGER"}, "DscConjunto": {"VARCHAR"},
    },
    "regiao": {
        "IdeConjunto": {"BIGINT", "INTEGER"}, "CodMunicipio": {"BIGINT", "INTEGER"},
        "Municipio": {"VARCHAR"}, "UF": {"VARCHAR"}, "Regiao": {"VARCHAR"},
    },
}

schema_resultados = []
for nome, caminho in DATASETS.items():
    descricao = {row[0]: row[1] for row in con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [str(caminho)]).fetchall()}
    colunas_faltantes = sorted(set(SCHEMA_ESPERADO[nome]) - set(descricao))
    tipos_incompativeis = sorted(
        f"{coluna}: esperado {sorted(tipos)}, encontrado {descricao[coluna]}"
        for coluna, tipos in SCHEMA_ESPERADO[nome].items()
        if coluna in descricao and descricao[coluna] not in tipos
    )
    schema_resultados.append({
        "dataset": nome,
        "colunas_identificadas": len(descricao),
        "colunas_faltantes": ", ".join(colunas_faltantes) or "0",
        "tipos_incompativeis": "; ".join(tipos_incompativeis) or "0",
    })

df_schema = pd.DataFrame(schema_resultados)
display(df_schema)

,dataset,colunas_identificadas,colunas_faltantes,tipos_incompativeis
0,continuidade,8,0,0
1,interrupcoes,15,0,0
2,limites,6,0,0
3,atributos,9,0,0
4,regiao,5,0,0


## 5. Validação de filtros de escopo

In [112]:
# Verifica anos, períodos e indicadores aceitos pelo contrato
escopo_queries = {
    "continuidade_ano": f"SELECT COUNT(*) FROM read_parquet(?) WHERE AnoIndice NOT BETWEEN {ANO_INICIO} AND {ANO_FIM}",
    "continuidade_periodo": "SELECT COUNT(*) FROM read_parquet(?) WHERE NumPeriodoIndice NOT BETWEEN 1 AND 12",
    "continuidade_indicador": "SELECT COUNT(*) FROM read_parquet(?) WHERE SigIndicador NOT IN ('DEC', 'FEC') OR SigIndicador IS NULL",
    "interrupcoes_ano": f"SELECT COUNT(*) FROM read_parquet(?) WHERE AnoIndice NOT BETWEEN {ANO_INICIO} AND {ANO_FIM}",
    "limites_ano": f"SELECT COUNT(*) FROM read_parquet(?) WHERE AnoIndice NOT BETWEEN {ANO_INICIO} AND {ANO_FIM}",
    "limites_indicador": "SELECT COUNT(*) FROM read_parquet(?) WHERE SigIndicador NOT IN ('DEC', 'FEC') OR SigIndicador IS NULL",
}
escopo_fontes = {
    "continuidade_ano": CONT_PATH, "continuidade_periodo": CONT_PATH, "continuidade_indicador": CONT_PATH,
    "interrupcoes_ano": INT_PATH, "limites_ano": LIM_PATH, "limites_indicador": LIM_PATH,
}
escopo_resultados = []
for regra, consulta in escopo_queries.items():
    falhas = int(con.execute(consulta, [str(escopo_fontes[regra])]).fetchone()[0])
    escopo_resultados.append({"regra": regra, "falhas": falhas})

df_escopo = pd.DataFrame(escopo_resultados)
display(df_escopo)

,regra,falhas
0,continuidade_ano,0
1,continuidade_periodo,0
2,continuidade_indicador,0
3,interrupcoes_ano,0
4,limites_ano,0
5,limites_indicador,0


## 6. Validação de nulos e conversões

In [117]:
# Campos críticos nulos e conversões que resultaram em NULL
nulos_queries = {
    "continuidade_ide_conjunto": (CONT_PATH, "IdeConjunto IS NULL"),
    "continuidade_valor": (CONT_PATH, "VlrIndiceEnviado IS NULL"),
    "interrupcoes_ide_conjunto": (INT_PATH, "IdeConjunto IS NULL"),
    "interrupcoes_inicio": (INT_PATH, "DatInicioInterrupcao IS NULL"),
    "interrupcoes_fim": (INT_PATH, "DatFimInterrupcao IS NULL"),
    "interrupcoes_duracao": (INT_PATH, "DuracaoHoras IS NULL"),
    "limites_ide_conjunto": (LIM_PATH, "IdeConjunto IS NULL"),
    "limites_valor": (LIM_PATH, "VlrLimite IS NULL"),
    "atributos_ide_conjunto": (ATR_PATH, "IdeConjunto IS NULL"),
    "regiao_ide_conjunto": (REG_PATH, "IdeConjunto IS NULL"),
}

nulos_resultados = []
for regra, (caminho, condicao) in nulos_queries.items():
    total, nulos = con.execute(
        f"SELECT COUNT(*), COUNT(*) FILTER (WHERE {condicao}) FROM read_parquet(?)",
        [str(caminho)],
    ).fetchone()
    percentual = round((nulos / total * 100) if total else 0, 2)
    nulos_resultados.append({"regra": regra, "total": total, "falhas": nulos, "percentual_falhas": percentual})

df_nulos = pd.DataFrame(nulos_resultados)
display(df_nulos.style.format({"percentual_falhas": "{:.2f} %"}))

,regra,total,falhas,percentual_falhas
0,continuidade_ide_conjunto,375082,0,0.00 %
1,continuidade_valor,375082,0,0.00 %
2,interrupcoes_ide_conjunto,44410820,0,0.00 %
3,interrupcoes_inicio,44410820,0,0.00 %
4,interrupcoes_fim,44410820,0,0.00 %
5,interrupcoes_duracao,44410820,0,0.00 %
6,limites_ide_conjunto,31844,0,0.00 %
7,limites_valor,31844,0,0.00 %
8,atributos_ide_conjunto,7569,0,0.00 %
9,regiao_ide_conjunto,15162,0,0.00 %


## 7. Validação de granularidade e duplicidades

In [101]:
# Duplicidades nas granularidades definidas pelo contrato
duplicidade_queries = {
    "continuidade": (CONT_PATH, "IdeConjunto, AnoIndice, NumPeriodoIndice, SigIndicador"),
    "limites": (LIM_PATH, "NumCNPJ, IdeConjunto, SigIndicador, AnoIndice"),
    "atributos": (ATR_PATH, "NumCNPJ, IdeConjunto"),
    "regiao": (REG_PATH, "IdeConjunto"),
}

duplicidade_resultados = []
for nome, (caminho, chave) in duplicidade_queries.items():
    consulta = f"""
        SELECT COUNT(*) FROM (
            SELECT {chave}
            FROM read_parquet(?)
            GROUP BY {chave}
            HAVING COUNT(*) > 1
        )
    """
    falhas = int(con.execute(consulta, [str(caminho)]).fetchone()[0])
    duplicidade_resultados.append({"dataset": nome, "falhas": falhas})

df_duplicidades = pd.DataFrame(duplicidade_resultados)
display(df_duplicidades)

,dataset,falhas
0,continuidade,0
1,limites,0
2,atributos,0
3,regiao,0


## 8. Validação de chaves e relacionamentos

In [102]:
# Diagnóstico complementar de relacionamentos
relacionamento_queries = {
    "continuidade_sem_regiao": f"""
        SELECT COUNT(*) FROM (
            SELECT DISTINCT c.IdeConjunto
            FROM read_parquet('{CONT_PATH}') c
            LEFT JOIN read_parquet('{REG_PATH}') r USING (IdeConjunto)
            WHERE r.IdeConjunto IS NULL
        )
    """,
    "interrupcoes_sem_regiao": f"""
        SELECT COUNT(*) FROM (
            SELECT DISTINCT i.IdeConjunto
            FROM read_parquet('{INT_PATH}') i
            LEFT JOIN read_parquet('{REG_PATH}') r USING (IdeConjunto)
            WHERE r.IdeConjunto IS NULL
        )
    """,
    "continuidade_sem_limite": f"""
        SELECT COUNT(*) FROM (
            SELECT DISTINCT c.IdeConjunto, c.SigIndicador, c.AnoIndice
            FROM read_parquet('{CONT_PATH}') c
            LEFT JOIN read_parquet('{LIM_PATH}') l
              ON c.IdeConjunto = l.IdeConjunto
             AND c.SigIndicador = l.SigIndicador
             AND c.AnoIndice = l.AnoIndice
            WHERE l.IdeConjunto IS NULL
        )
    """,
}

relacionamento_resultados = []
for regra, consulta in relacionamento_queries.items():
    falhas = int(con.execute(consulta).fetchone()[0])
    relacionamento_resultados.append({"regra": regra, "falhas": falhas})

df_relacionamentos = pd.DataFrame(relacionamento_resultados)
display(df_relacionamentos)

,regra,falhas
0,continuidade_sem_regiao,4
1,interrupcoes_sem_regiao,4
2,continuidade_sem_limite,64


## 9. Validações complementares por dataset

In [103]:
# Validações complementares por dataset
especificas_queries = {
    "continuidade_valor_negativo": (CONT_PATH, "VlrIndiceEnviado < 0"),
    "interrupcoes_duracao_negativa": (INT_PATH, "DuracaoHoras < 0"),
    "interrupcoes_datas_invertidas": (INT_PATH, "DatFimInterrupcao < DatInicioInterrupcao"),
    "interrupcoes_consumidores_negativos": (INT_PATH, "NumConsumidorConjunto < 0"),
    "regiao_uf_vazia": (REG_PATH, "UF IS NULL OR TRIM(UF) = ''"),
    "regiao_regiao_invalida": (REG_PATH, "Regiao NOT IN ('NORTE', 'NORDESTE', 'CENTRO-OESTE', 'SUDESTE', 'SUL', 'NAO INFORMADO')"),
}

especificas_resultados = []
for regra, (caminho, condicao) in especificas_queries.items():
    falhas = int(con.execute(f"SELECT COUNT(*) FROM read_parquet(?) WHERE {condicao}", [str(caminho)]).fetchone()[0])
    especificas_resultados.append({"regra": regra, "falhas": falhas})

df_especificas = pd.DataFrame(especificas_resultados)
display(df_especificas)

,regra,falhas
0,continuidade_valor_negativo,0
1,interrupcoes_duracao_negativa,0
2,interrupcoes_datas_invertidas,0
3,interrupcoes_consumidores_negativos,0
4,regiao_uf_vazia,29
5,regiao_regiao_invalida,0


## 10. Problemas encontrados e ações recomendadas

In [104]:
# Detalha as regras com falhas e mede o impacto relativo
impacto_queries = {
    "continuidade_sem_regiao": (f"SELECT COUNT(DISTINCT IdeConjunto) FROM read_parquet('{CONT_PATH}')", "A ausência regional é tratada como 'NÃO INFORMADO' no modelo e afeta apenas o enriquecimento geográfico."),
    "interrupcoes_sem_regiao": (f"SELECT COUNT(DISTINCT IdeConjunto) FROM read_parquet('{INT_PATH}')", "A ausência regional é tratada como 'NÃO INFORMADO' no modelo e afeta apenas o enriquecimento geográfico."),
    "continuidade_sem_limite": (f"SELECT COUNT(*) FROM (SELECT DISTINCT IdeConjunto, SigIndicador, AnoIndice FROM read_parquet('{CONT_PATH}'))", "A combinação sem limite não pode ser usada em comparação regulatória."),
    "regiao_uf_vazia": (f"SELECT COUNT(*) FROM read_parquet('{REG_PATH}')", "A UF vazia reduz o detalhamento geográfico; o modelo usa 'NÃO INFORMADO' quando necessário."),
    "continuidade_valor_negativo": (f"SELECT COUNT(*) FROM read_parquet('{CONT_PATH}')", "Valor negativo viola o domínio definido para o indicador."),
    "interrupcoes_duracao_negativa": (f"SELECT COUNT(*) FROM read_parquet('{INT_PATH}')", "Duração negativa indica inconsistência temporal."),
    "interrupcoes_datas_invertidas": (f"SELECT COUNT(*) FROM read_parquet('{INT_PATH}')", "Data final anterior ao início indica inconsistência temporal."),
    "interrupcoes_consumidores_negativos": (f"SELECT COUNT(*) FROM read_parquet('{INT_PATH}')", "Quantidade negativa de consumidores viola o domínio do campo."),
    "regiao_regiao_invalida": (f"SELECT COUNT(*) FROM read_parquet('{REG_PATH}')", "Região fora do domínio impede a classificação geográfica correta."),
}

problemas = []
for tabela in [df_relacionamentos, df_especificas]:
    for _, linha in tabela[tabela["falhas"] > 0].iterrows():
        regra = linha["regra"]
        if regra not in impacto_queries:
            continue
        consulta_total, justificativa = impacto_queries[regra]
        total_referencia = int(con.execute(consulta_total).fetchone()[0])
        ocorrencias = int(linha["falhas"])
        problemas.append({
            "regra": regra,
            "ocorrencias": ocorrencias,
            "total_referencia": total_referencia,
            "percentual_afetado": round(ocorrencias / total_referencia * 100, 2) if total_referencia else 0,
            "justificativa": justificativa,
        })

if problemas:
    df_problemas = pd.DataFrame(problemas)
    display(df_problemas.style.format({"percentual_afetado": "{:.2f} %"}))
else:
    display(pd.DataFrame({"resultado": ["Nenhuma falha foi encontrada nas validações complementares."]}))


,regra,ocorrencias,total_referencia,percentual_afetado,justificativa
0,continuidade_sem_regiao,4,3824,0.10 %,A ausência regional é tratada como 'NÃO INFORMADO' no modelo e afeta apenas o enriquecimento geográfico.
1,interrupcoes_sem_regiao,4,3766,0.11 %,A ausência regional é tratada como 'NÃO INFORMADO' no modelo e afeta apenas o enriquecimento geográfico.
2,continuidade_sem_limite,64,31292,0.20 %,A combinação sem limite não pode ser usada em comparação regulatória.
3,regiao_uf_vazia,29,15162,0.19 %,A UF vazia reduz o detalhamento geográfico; o modelo usa 'NÃO INFORMADO' quando necessário.


As falhas identificadas não impedem o consumo da camada interim, com valores inferiores a 1%, sendo documentadas apenas para fins de auditoria e melhoria contínua.